In [7]:
import pandas as pd
import numpy as np

from tensorflow.keras.preprocessing.sequence import pad_sequences

from sklearn.model_selection import train_test_split

In [2]:
route_dataset_df = pd.read_pickle(
    "../data/processed/route_dataset.pkl"
)

print(route_dataset_df.shape)

(19647, 15)


In [3]:
print(type(route_dataset_df.iloc[0]["Planned Route"]))
print(route_dataset_df.iloc[0]["Planned Route"])

<class 'list'>
[0, 1, 2, 3, 3, 0, 1]


In [4]:
MAX_ROUTE_LENGTH = route_dataset_df[
    "Planned Route"
].apply(len).max()

print(MAX_ROUTE_LENGTH)

36


In [8]:
route_sequences = pad_sequences(
    route_dataset_df["Planned Route"],
    maxlen=MAX_ROUTE_LENGTH,
    padding="post",
    truncating="post",
    value=0
)

print(route_sequences.shape)

(19647, 36)


In [10]:
max_stop_id = max([
    max(route)
    for route in route_dataset_df["Planned Route"]
])

VOCAB_SIZE = max_stop_id + 1

print(VOCAB_SIZE)

10864


In [11]:
driver_input = route_dataset_df[
    "Driver ID"
].values

numerical_features = route_dataset_df[
    [
        "Country",
        "Week ID",
        "Planned Stop Count",
        "Planned Distance"
    ]
].values

y = route_dataset_df[
    "RQS"
].values

print(driver_input.shape)
print(numerical_features.shape)
print(y.shape)

(19647,)
(19647, 4)
(19647,)


In [12]:
(
    route_train,
    route_test,

    driver_train,
    driver_test,

    num_train,
    num_test,

    y_train,
    y_test

) = train_test_split(

    route_sequences,

    driver_input,

    numerical_features,

    y,

    test_size=0.2,

    random_state=42

)

print(route_train.shape)
print(driver_train.shape)
print(num_train.shape)
print(y_train.shape)

(15717, 36)
(15717,)
(15717, 4)
(15717,)


In [13]:
from tensorflow.keras.layers import (
    Input,
    Embedding,
    LSTM,
    Dense,
    Flatten,
    Concatenate
)

from tensorflow.keras.models import Model


# -----------------------------
# Driver Branch
# -----------------------------

driver_layer = Input(
    shape=(1,),
    name="Driver_Input"
)

driver_embedding = Embedding(
    input_dim=route_dataset_df["Driver ID"].max() + 1,
    output_dim=32
)(
    driver_layer
)

driver_embedding = Flatten()(
    driver_embedding
)


# -----------------------------
# Route Branch
# -----------------------------

route_layer = Input(
    shape=(MAX_ROUTE_LENGTH,),
    name="Route_Input"
)

route_embedding = Embedding(
    input_dim=VOCAB_SIZE,
    output_dim=64,
    mask_zero=True
)(
    route_layer
)

route_lstm = LSTM(
    64
)(
    route_embedding)


# -----------------------------
# Numerical Features Branch
# -----------------------------

numerical_layer = Input(
    shape=(4,),
    name="Numerical_Input"
)

numerical_dense = Dense(
    32,
    activation="relu"
)(
    numerical_layer
)


# -----------------------------
# Merge
# -----------------------------

merged = Concatenate()(
    [
        driver_embedding,
        route_lstm,
        numerical_dense
    ]
)

dense1 = Dense(
    128,
    activation="relu"
)(
    merged
)

dense2 = Dense(
    64,
    activation="relu"
)(
    dense1
)

output = Dense(
    1,
    activation="sigmoid"
)(
    dense2)


model = Model(
    inputs=[
        driver_layer,
        route_layer,
        numerical_layer
    ],
    outputs=output
)

In [14]:
model.compile(
    optimizer="adam",
    loss="mse",
    metrics=["mae"]
)

model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ Driver_Input        │ (None, 1)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Route_Input         │ (None, 36)        │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding           │ (None, 1, 32)     │     12,800 │ Driver_Input[0][… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding_1         │ (None, 36, 64)    │    695,296 │ Route_Input[0][0] │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ not_equal           │ (None, 36)        │          0 │ Route_Input[0][0] │
│ (NotEqual)          │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Numerical_Input     │ (None, 4)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ flatten (Flatten)   │ (None, 32)        │          0 │ embedding[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm (LSTM)         │ (None, 64)        │     33,024 │ embedding_1[0][0… │
│                     │                   │            │ not_equal[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense (Dense)       │ (None, 32)        │        160 │ Numerical_Input[… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate         │ (None, 128)       │          0 │ flatten[0][0],    │
│ (Concatenate)       │                   │            │ lstm[0][0],       │
│                     │                   │            │ dense[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_1 (Dense)     │ (None, 128)       │     16,512 │ concatenate[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_2 (Dense)     │ (None, 64)        │      8,256 │ dense_1[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_3 (Dense)     │ (None, 1)         │         65 │ dense_2[0][0]     │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 766,113 (2.92 MB)

 Trainable params: 766,113 (2.92 MB)

 Non-trainable params: 0 (0.00 B)

In [15]:
history = model.fit(

    [
        driver_train,
        route_train,
        num_train
    ],

    y_train,

    validation_split=0.2,

    epochs=20,

    batch_size=512,

    verbose=1

)

Epoch 1/20
25/25 ━━━━━━━━━━━━━━━━━━━━ 3s 77ms/step - loss: 0.0507 - mae: 0.1611 - val_loss: 0.0421 - val_mae: 0.1517
Epoch 2/20
25/25 ━━━━━━━━━━━━━━━━━━━━ 2s 76ms/step - loss: 0.0350 - mae: 0.1431 - val_loss: 0.0274 - val_mae: 0.1302
Epoch 3/20
25/25 ━━━━━━━━━━━━━━━━━━━━ 2s 80ms/step - loss: 0.0243 - mae: 0.1172 - val_loss: 0.0227 - val_mae: 0.1108
Epoch 4/20
25/25 ━━━━━━━━━━━━━━━━━━━━ 2s 84ms/step - loss: 0.0201 - mae: 0.1020 - val_loss: 0.0210 - val_mae: 0.1074
Epoch 5/20
25/25 ━━━━━━━━━━━━━━━━━━━━ 2s 80ms/step - loss: 0.0155 - mae: 0.0896 - val_loss: 0.0201 - val_mae: 0.1027
Epoch 6/20
25/25 ━━━━━━━━━━━━━━━━━━━━ 2s 77ms/step - loss: 0.0119 - mae: 0.0777 - val_loss: 0.0206 - val_mae: 0.1039
Epoch 7/20
25/25 ━━━━━━━━━━━━━━━━━━━━ 2s 84ms/step - loss: 0.0095 - mae: 0.0686 - val_loss: 0.0204 - val_mae: 0.1015
Epoch 8/20
25/25 ━━━━━━━━━━━━━━━━━━━━ 2s 76ms/step - loss: 0.0079 - mae: 0.0620 - val_loss: 0.0215 - val_mae: 0.1026
Epoch 9/20
25/25 ━━━━━━━━━━━━━━━━━━━━ 2s 83ms/step - loss: 0.006

In [16]:
y_pred = model.predict(
    [
        driver_test,
        route_test,
        num_test
    ]
)

123/123 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step


In [17]:
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

import numpy as np

mae = mean_absolute_error(
    y_test,
    y_pred
)

mse = mean_squared_error(
    y_test,
    y_pred
)

rmse = np.sqrt(
    mse
)

r2 = r2_score(
    y_test,
    y_pred
)

print("="*60)
print("LSTM REGRESSION RESULTS")
print("="*60)

print(f"MAE  : {mae:.4f}")
print(f"RMSE : {rmse:.4f}")
print(f"R²   : {r2:.4f}")

LSTM REGRESSION RESULTS
MAE  : 0.1054
RMSE : 0.1558
R²   : 0.3258


In [18]:
model.save(
    "../outputs/models/lstm_regression.keras"
)

print(
    "LSTM model saved successfully!"
)

LSTM model saved successfully!
